<a href="https://colab.research.google.com/github/shanshiny06/econ3916-portfolio/blob/main/Problem%20Set%202/Econ_3916_PS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

   # Econ 3916 PS2: The Survey That Wasn't Random
   ## Phase 1: Simulate the Population You Cannot See

In [2]:
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy import stats

rng = np.random.default_rng(3916)  # fixed seed so a grader can rerun and get the same numbers
N = 50_000             # users per period

def build_population(base, rng):
    """Simulate one period's users with the covariates and satisfaction score."""
    # lognormal's mean and sigma are those of the underlying normal, not of the minutes
    engagement_minutes = rng.lognormal(mean=3.5, sigma=0.6, size=N)
    engagement_z = (engagement_minutes - engagement_minutes.mean()) / engagement_minutes.std()
    had_support_ticket = rng.binomial(1, 0.22, size=N)   # Bernoulli(0.22)
    # latent satisfaction: engagement raises it, a support ticket lowers it
    s = base + 0.35 * engagement_z - 0.60 * had_support_ticket + rng.normal(0, 0.75, size=N)
    score = np.clip(np.round(s), 1, 5)           # five-point survey scale
    return pd.DataFrame({"engagement_minutes": engagement_minutes, "engagement_z": engagement_z,
                         "had_support_ticket": had_support_ticket, "s": s, "score": score})

pre = build_population(3.432, rng)  # before the redesign
post = build_population(3.532, rng) # after the redesign

# The truth we keep on file: 3.432 - 0.60 * 0.22 = 3.30, and 3.532 - 0.132 = 3.40
true_pre, true_post = pre["s"].mean(), post["s"].mean()
print(f"True mean (latent s):   pre {true_pre:.3f}   post {true_post:.3f}   gap {true_post - true_pre:.3f}")
print(f"True mean (1-5 score):  pre {pre['score'].mean():.3f}   post {post['score'].mean():.3f}")

True mean (latent s):   pre 3.295   post 3.406   gap 0.111
True mean (1-5 score):  pre 3.289   post 3.393


In [3]:
def respond(df, a, b, rng):
    """Return True for each user who answers the survey."""
    # response probability rises with satisfaction itself, plus the two covariates
    p = expit(a + b * (df["s"] - 3.3) + 0.30 * df["engagement_z"] - 0.25 * df["had_support_ticket"])
    return rng.uniform(size=len(df)) < p     # a user answers if a uniform draw falls below p

pre["responded"] = respond(pre, a=-1.0, b=0.45, rng=rng)     # before: mildly selective
post["responded"] = respond(post, a=-2.75, b=2.00, rng=rng)  # after: far more selective

# What the PM sees: the mean score among respondents only
obs_pre = pre.loc[pre["responded"], "score"].mean()
obs_post = post.loc[post["responded"], "score"].mean()
print(f"Observed survey mean:  pre {obs_pre:.3f}   post {obs_post:.3f}   apparent gap {obs_post - obs_pre:.3f}")
print(f"Response rate:         pre {pre['responded'].mean():.1%}   post {post['responded'].mean():.1%}")

Observed survey mean:  pre 3.593   post 4.317   apparent gap 0.724
Response rate:         pre 27.2%   post 15.5%


This is non-response bias, a form of selection bias from Chapter 6: who answers the survey depends on the very thing we are measuring. After the redesign, the response probability rises much more steeply with satisfaction (b goes from 0.45 to 2.00), so satisfied users are heavily over-represented among respondents. The survey mean jumps from about 3.59 to 4.32, an apparent gain of about 0.72 points, while the true gain in the population is only 0.10 (0.111 in this simulation).

A larger survey would not fix this. Bias does not shrink with sample size: surveying more users through the same prompt draws more people from the same skewed pool of respondents, so the estimate would converge more precisely on the wrong number, about 4.3 instead of 3.4. Only changing who responds, not how many, can remove the bias.

Note the response rates: about 27% answered before the redesign but only about 15% after. The survey that produced the better number heard from fewer people.

## Phase 2: Quantify What You Actually Know

In [4]:
def bootstrap_manual(data, stat, n_boot=10_000, seed=3916):
    """Resample the data with replacement n_boot times and return the statistic from each resample."""
    rng_boot = np.random.default_rng(seed)
    boot = np.empty(n_boot)
    for i in range(n_boot):
        resample = rng_boot.choice(data, size=len(data), replace=True)   # same size, with replacement
        boot[i] = stat(resample)
    return boot

# The survey data the PM actually has: scores of respondents only
resp_pre = pre.loc[pre["responded"], "score"].to_numpy()
resp_post = post.loc[post["responded"], "score"].to_numpy()
samples = {"pre": resp_pre, "post": resp_post}
statistics = {"mean": np.mean, "median": np.median}

rows = []
for period, data in samples.items():
    for name, stat in statistics.items():
        boot = bootstrap_manual(data, stat)
        low, high = np.percentile(boot, [2.5, 97.5])    # percentile interval: middle 95% of the resamples
        rows.append({"period": period, "statistic": name, "estimate": stat(data),
                     "manual_low": low, "manual_high": high})
boot_table = pd.DataFrame(rows)
boot_table.round(4)

,period,statistic,estimate,manual_low,manual_high
0,pre,mean,3.5932,3.5785,3.6079
1,pre,median,4.0000,4.0000,4.0000
2,post,mean,4.3168,4.3018,4.3319
3,post,median,4.0000,4.0000,4.0000


In [5]:
import warnings

for i, row in boot_table.iterrows():
    data = samples[row["period"]]
    stat = statistics[row["statistic"]]
    for method in ["percentile", "BCa"]:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")   # BCa warns when every resample gives the same median
            ci = stats.bootstrap((data,), stat, method=method, n_resamples=10_000,
                                 batch=100, random_state=3916).confidence_interval
        boot_table.loc[i, f"{method}_low"] = ci.low
        boot_table.loc[i, f"{method}_high"] = ci.high

boot_table.round(4)

,period,statistic,estimate,manual_low,manual_high,percentile_low,percentile_high,BCa_low,BCa_high
0,pre,mean,3.5932,3.5785,3.6079,3.5785,3.6080,3.5783,3.6078
1,pre,median,4.0000,4.0000,4.0000,4.0000,4.0000,NaN,NaN
2,post,mean,4.3168,4.3018,4.3319,4.3020,4.3318,4.3019,4.3317
3,post,median,4.0000,4.0000,4.0000,4.0000,4.0000,NaN,NaN


The manual loop and scipy.stats.bootstrap agree to about three decimal places; the small differences come from different random resamples, not from different methods.

For the mean, the percentile and BCa intervals are almost identical (post-redesign: about 4.30 to 4.33). With thousands of respondents, the bootstrap distribution of the mean is nearly symmetric and unbiased, so BCa's corrections for bias and skew have almost nothing to correct.

For the median, the two methods differ completely. The percentile interval is [4, 4], and BCa cannot be computed at all (scipy returns NaN). On a five-point scale, almost every resample has a median of exactly 4, so the bootstrap distribution has no spread, and BCa's acceleration term divides by zero. The median is simply not an informative summary of a five-point score here.

Most importantly, the post-redesign interval is very narrow, about 4.30 to 4.33, yet the true population mean is 3.39. The interval is precise but centred on the wrong number: a bootstrap measures sampling noise among the people who answered, and says nothing about who did not answer.

In [6]:
# Each repeated survey invites a random 2,000 users from the post-redesign population;
# who answers follows the same response mechanism as in Step 1.2
p_post = expit(-2.75 + 2.00 * (post["s"] - 3.3) + 0.30 * post["engagement_z"]
               - 0.25 * post["had_support_ticket"]).to_numpy()
score_post = post["score"].to_numpy()

truth_population = score_post.mean()                               # what the PM wants to know
truth_respondents = (p_post * score_post).sum() / p_post.sum()     # what the survey actually estimates

n_surveys, n_invited = 1000, 2000
rng_cov = np.random.default_rng(2026)
hits = {(m, t): 0 for m in ["percentile", "BCa"] for t in ["population", "respondents"]}

for _ in range(n_surveys):
    invited = rng_cov.choice(N, size=n_invited, replace=False)
    answered = rng_cov.uniform(size=n_invited) < p_post[invited]
    y = score_post[invited][answered]
    for method in ["percentile", "BCa"]:
        ci = stats.bootstrap((y,), np.mean, method=method, n_resamples=999,
                             random_state=rng_cov).confidence_interval
        hits[(method, "population")] += ci.low <= truth_population <= ci.high
        hits[(method, "respondents")] += ci.low <= truth_respondents <= ci.high

rows = []
for (method, target), h in hits.items():
    c = h / n_surveys
    rows.append({"interval": method, "truth": target, "coverage": c,
                 "MC_SE": np.sqrt(c * (1 - c) / n_surveys),      # Monte Carlo standard error
                 "gap_from_95": c - 0.95})
coverage_table = pd.DataFrame(rows)
print(f"Truth (population mean score): {truth_population:.3f}")
print(f"Truth (expected respondent mean): {truth_respondents:.3f}")
coverage_table.round(4)

Truth (population mean score): 3.393
Truth (expected respondent mean): 4.317


,interval,truth,coverage,MC_SE,gap_from_95
0,percentile,population,0.000,0.0000,-0.950
1,percentile,respondents,0.951,0.0068,0.001
2,BCa,population,0.000,0.0000,-0.950
3,BCa,respondents,0.954,0.0066,0.004


Against the true population mean (3.39), both intervals have a coverage of 0 out of 1,000 surveys. This is not a near miss: every single interval sits around 4.3, about 0.9 points above the truth. The selection bias is far larger than the width of any interval.

Against the quantity the survey actually estimates, the expected mean among respondents (4.32), coverage is 95.1% for the percentile interval and 95.4% for BCa, each with a Monte Carlo standard error of about 0.7 percentage points. Both gaps from the nominal 95% (0.1 and 0.4 points) are smaller than one Monte Carlo standard error, so they are not a finding: we cannot say either method over- or under-covers, or that one is better than the other.

The bootstrap does exactly what it promises, correctly quantifying sampling uncertainty, but for the wrong target. A 95% interval is a statement about repeated sampling from the respondents, not about the users who never answered.

## Phase 3: Test Something Worth Testing

In [7]:
# Welch's two-sample t-test on the survey means, post minus pre
t_test = stats.ttest_ind(resp_post, resp_pre, equal_var=False)

# Cohen's d: the observed gap in units of the pooled standard deviation
n_post, n_pre = len(resp_post), len(resp_pre)
s_pooled = np.sqrt(((n_post - 1) * resp_post.var(ddof=1) + (n_pre - 1) * resp_pre.var(ddof=1))
                   / (n_post + n_pre - 2))
d_observed = (resp_post.mean() - resp_pre.mean()) / s_pooled

# The same effect size computed from the full populations, which the PM cannot see
sd_score = np.std(np.concatenate([pre["score"], post["score"]]), ddof=1)
true_gap = post["score"].mean() - pre["score"].mean()
d_true = true_gap / sd_score

print(f"Survey gap: {resp_post.mean() - resp_pre.mean():.3f} points   t = {t_test.statistic:.2f}   p = {t_test.pvalue:.3g}")
print(f"Cohen's d (survey):  {d_observed:.3f}")
print(f"True gap: {true_gap:.3f} points   Cohen's d (population): {d_true:.3f}")

Survey gap: 0.724 points   t = 67.80   p = 0
Cohen's d (survey):  0.900
True gap: 0.104 points   Cohen's d (population): 0.117


The test is overwhelmingly significant: Welch's t is about 67.8 and the p-value is effectively zero. The effect size is large too: Cohen's d is about 0.90, which would usually count as a large effect.

But statistical significance is not the issue here. A t-test asks whether the gap between the two samples could be due to random sampling noise. It assumes each sample is a fair draw from the population it describes. These samples are not: the post-redesign respondents were selected far more heavily on satisfaction. The test is therefore answering a question nobody asked, whether these two groups of respondents differ, and answering it correctly. With thousands of respondents, even a gap produced entirely by who chose to answer will be "significant."

The true improvement in the population is about 0.10 points, a Cohen's d of about 0.12, roughly one seventh of what the survey suggests. No p-value can detect or remove selection bias; only the design of the data collection can.

In [8]:
from statsmodels.stats.power import TTestIndPower
power_analysis = TTestIndPower()

# (a) Respondents per group needed to detect a true 0.1-point improvement with 80% power
d_target = 0.1 / sd_score
n_required = power_analysis.solve_power(effect_size=d_target, alpha=0.05, power=0.80,
                                        ratio=1.0, alternative="two-sided")

# (b) Smallest effect detectable with 80% power at the sample sizes the survey actually gets
d_detectable = power_analysis.solve_power(nobs1=n_post, ratio=n_pre / n_post, alpha=0.05,
                                          power=0.80, alternative="two-sided")

print(f"SD of the 1-5 score: {sd_score:.3f}")
print(f"(a) A 0.1-point improvement is d = {d_target:.3f}; it needs {n_required:,.0f} respondents per group")
print(f"(b) With {n_post:,} and {n_pre:,} respondents, the smallest detectable effect is "
      f"d = {d_detectable:.3f}, about {d_detectable * sd_score:.3f} points")

SD of the 1-5 score: 0.889
(a) A 0.1-point improvement is d = 0.112; it needs 1,242 respondents per group
(b) With 7,749 and 13,578 respondents, the smallest detectable effect is d = 0.040, about 0.035 points


In [9]:
# Each period's response probability, from the mechanism in Step 1.2
p_pre = expit(-1.0 + 0.45 * (pre["s"] - 3.3) + 0.30 * pre["engagement_z"]
              - 0.25 * pre["had_support_ticket"]).to_numpy()
score_pre = pre["score"].to_numpy()

# Re-run the pre/post survey many times at each invitation size, and compare the
# estimated gap with the true gap of about 0.10
rng_n = np.random.default_rng(7)
rows = []
for n_invited in [500, 2000, 8000, 32000, 50000]:
    gaps = []
    for _ in range(300):
        idx_pre = rng_n.choice(N, size=n_invited, replace=False)
        idx_post = rng_n.choice(N, size=n_invited, replace=False)
        ans_pre = rng_n.uniform(size=n_invited) < p_pre[idx_pre]
        ans_post = rng_n.uniform(size=n_invited) < p_post[idx_post]
        gaps.append(score_post[idx_post][ans_post].mean() - score_pre[idx_pre][ans_pre].mean())
    gaps = np.array(gaps)
    rows.append({"n_invited_per_period": n_invited,
                 "mean_estimated_gap": gaps.mean(),
                 "sd_of_estimate": gaps.std(ddof=1),                      # sampling noise
                 "bias": gaps.mean() - true_gap,                          # systematic error
                 "rmse": np.sqrt(np.mean((gaps - true_gap) ** 2))})       # total error
size_table = pd.DataFrame(rows)
size_table.round(4)

,n_invited_per_period,mean_estimated_gap,sd_of_estimate,bias,rmse
0,500,0.7202,0.1041,0.6162,0.6249
1,2000,0.7160,0.0551,0.6119,0.6144
2,8000,0.7168,0.0257,0.6127,0.6133
3,32000,0.7169,0.0118,0.6128,0.6129
4,50000,0.7166,0.0084,0.6126,0.6126


For a true improvement of 0.1 points (d of about 0.11, using the score's SD of about 0.89), a follow-up survey needs about 1,240 respondents per group for 80% power. The survey already gets far more than that: with about 7,750 and 13,580 respondents, it could detect an effect as small as about 0.035 points. Power is not the problem.

Would a larger sample make the estimate better? No. The simulation re-runs the survey 300 times at each size, from 500 to all 50,000 users invited per period. As the sample grows, the standard deviation of the estimated gap falls from about 0.10 to under 0.01, so the estimate becomes very precise. But its average stays fixed at about 0.72, a bias of about 0.61 points above the true gap of 0.10, and the bias does not shrink at all. The total error (RMSE) falls from 0.62 only to 0.61 and then stops: beyond a few thousand invitations, all of the remaining error is bias.

A larger sample buys a more precise estimate of the wrong number. Because response depends on satisfaction itself, the only fix is to change how respondents are selected, not how many there are.

In [10]:
"""
Inverse-probability weighting (IPW) for survey non-response.

Expected columns in the sample-frame DataFrame (ALL sampled users,
respondents and non-respondents alike):
    satisfaction        -- observed only where responded == 1 (NaN otherwise)
    engagement_minutes  -- known for everyone (product logs)
    had_support_ticket  -- known for everyone (0/1)
    responded           -- 1 if the user answered the survey, else 0

Replace `simulate_population()` with your own DataFrame if you have one;
`ipw_mean()` and `bootstrap_ipw()` only need the four columns above.
"""
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

COVARIATES = ["engagement_minutes", "had_support_ticket"]


# --------------------------------------------------------------------------
# Simulation matching the described mechanism: P(respond) rises with
# satisfaction ITSELF (MNAR). Covariates affect satisfaction, not response
# directly.
# --------------------------------------------------------------------------
def simulate_population(n=20_000, seed=0):
    rng = np.random.default_rng(seed)
    engagement_minutes = rng.lognormal(mean=3.0, sigma=0.6, size=n)
    p_ticket = 1 / (1 + np.exp(-(-0.5 - 0.03 * (engagement_minutes - 20))))
    had_support_ticket = rng.binomial(1, p_ticket)
    satisfaction = (
        5.0
        + 0.04 * engagement_minutes
        - 1.2 * had_support_ticket
        + rng.normal(0, 1.5, size=n)
    )
    p_respond = 1 / (1 + np.exp(-(-1.0 + 0.8 * (satisfaction - satisfaction.mean()))))
    responded = rng.binomial(1, p_respond)

    df = pd.DataFrame({
        "satisfaction_true": satisfaction,  # oracle column, unknowable in practice
        "engagement_minutes": engagement_minutes,
        "had_support_ticket": had_support_ticket,
        "responded": responded,
    })
    df["satisfaction"] = np.where(responded == 1, satisfaction, np.nan)
    return df


# --------------------------------------------------------------------------
# IPW estimator
# --------------------------------------------------------------------------
def fit_response_propensity(df):
    """Logistic model of P(responded = 1 | covariates), fit on the full frame."""
    X = df[COVARIATES].to_numpy(dtype=float)
    X = (X - X.mean(axis=0)) / X.std(axis=0)  # scaling helps the solver
    # Effectively unpenalised; C=1e10 works across old and new sklearn versions
    model = LogisticRegression(C=1e10, max_iter=1000)
    model.fit(X, df["responded"].to_numpy())
    return model.predict_proba(X)[:, 1]


def ipw_mean(df, clip=1e-3):
    """Hajek (self-normalised) IPW mean of satisfaction among respondents."""
    p_hat = np.clip(fit_response_propensity(df), clip, 1.0)
    resp = df["responded"].to_numpy() == 1
    w = 1.0 / p_hat[resp]
    y = df.loc[resp, "satisfaction"].to_numpy()
    return np.sum(w * y) / np.sum(w)


def bootstrap_ipw(df, n_boot=1000, alpha=0.05, seed=1):
    """
    Nonparametric bootstrap: resample the WHOLE frame (respondents and
    non-respondents) and refit the propensity model each time, so the
    interval reflects uncertainty in the estimated weights too.
    """
    rng = np.random.default_rng(seed)
    n = len(df)
    estimates = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        estimates[b] = ipw_mean(df.iloc[idx].reset_index(drop=True))
    lo, hi = np.quantile(estimates, [alpha / 2, 1 - alpha / 2])
    return lo, hi, estimates


# --------------------------------------------------------------------------
if __name__ == "__main__":
    df = simulate_population()

    true_mean = df["satisfaction_true"].mean()
    naive_mean = df["satisfaction"].mean()  # pandas skips NaN
    ipw_est = ipw_mean(df)
    lo, hi, _ = bootstrap_ipw(df, n_boot=500)

    print(f"Response rate        : {df['responded'].mean():.3f}")
    print(f"True population mean : {true_mean:.3f}")
    print(f"Naive respondent mean: {naive_mean:.3f}  (bias {naive_mean - true_mean:+.3f})")
    print(f"IPW estimate         : {ipw_est:.3f}  (bias {ipw_est - true_mean:+.3f})")
    print(f"95% bootstrap CI     : [{lo:.3f}, {hi:.3f}]")
    print(f"CI covers truth?     : {lo <= true_mean <= hi}")
    print(f"Share of naive bias removed: "
          f"{1 - (ipw_est - true_mean) / (naive_mean - true_mean):.1%}")

Response rate        : 0.321
True population mean : 5.526
Naive respondent mean: 6.795  (bias +1.270)
IPW estimate         : 6.509  (bias +0.983)
95% bootstrap CI     : [6.472, 6.544]
CI covers truth?     : False
Share of naive bias removed: 22.6%


## Phase 4: AI Expansion — The Co-Pilot Era
### Task 4.1: The weighting attempt

In [12]:
"""
IPW non-response correction for `score`, run on the existing `pre` and `post`
DataFrames (no simulation). Paste into the notebook after `pre` and `post` exist.

Columns used: engagement_minutes, had_support_ticket, score, responded.
`engagement_z` is a rescaling of engagement_minutes, so it is left out
(including both would make the design matrix collinear).
`s` is never used: it is the latent quantity driving response and would not
be observable in a real survey.
"""
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

COVARIATES = ["engagement_minutes", "had_support_ticket"]
N_BOOT = 1000
ALPHA = 0.05


def response_propensity(df):
    """Fit P(responded | covariates) on the full frame; return fitted probabilities."""
    X = df[COVARIATES].astype(float).to_numpy()
    X = (X - X.mean(axis=0)) / X.std(axis=0)
    r = df["responded"].astype(int).to_numpy()
    model = LogisticRegression(C=1e10, max_iter=1000)  # effectively unpenalised
    model.fit(X, r)
    return model.predict_proba(X)[:, 1]


def ipw_mean(df, clip=1e-3):
    """Hajek IPW mean of score, using respondents' scores only."""
    p = np.clip(response_propensity(df), clip, 1.0)
    resp = df["responded"].astype(bool).to_numpy()
    w = 1.0 / p[resp]
    y = df["score"].to_numpy(dtype=float)[resp]
    return np.sum(w * y) / np.sum(w)


def bootstrap_ipw(df, rng, n_boot=N_BOOT, alpha=ALPHA):
    """Resample the whole frame and refit the response model in every resample."""
    n = len(df)
    est = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        est[b] = ipw_mean(df.iloc[idx])
    lo, hi = np.quantile(est, [alpha / 2, 1 - alpha / 2])
    return lo, hi, est


# One generator, used for pre first and then post, so the run is reproducible.
rng = np.random.default_rng(3916)

rows = []
for name, df in [("pre", pre), ("post", post)]:
    resp = df["responded"].astype(bool)
    observed = df.loc[resp, "score"].mean()
    true = df["score"].mean()  # full-population mean, known only in simulation
    est = ipw_mean(df)
    lo, hi, _ = bootstrap_ipw(df, rng)
    rows.append({
        "wave": name,
        "response_rate": resp.mean(),
        "observed_mean": observed,
        "ipw_estimate": est,
        "ci_low": lo,
        "ci_high": hi,
        "true_mean": true,
        "naive_bias": observed - true,
        "ipw_bias": est - true,
        "bias_removed_share": 1 - (est - true) / (observed - true),
        "ci_covers_truth": lo <= true <= hi,
    })

results = pd.DataFrame(rows).set_index("wave")
pd.set_option("display.float_format", "{:.3f}".format)
print(results.T)

# Expected sanity checks against the figures you quoted:
#   pre : observed 3.593, true 3.289
#   post: observed 4.317, true 3.393
num = ["true_mean", "observed_mean", "ipw_estimate"]
change = results.loc["post", num] - results.loc["pre", num]
print(f"\nPre-to-post change  true: {change['true_mean']:.3f}  "
      f"naive: {change['observed_mean']:.3f}  IPW: {change['ipw_estimate']:.3f}")

wave                  pre   post
response_rate       0.272  0.155
observed_mean       3.593  4.317
ipw_estimate        3.462  4.144
ci_low              3.447  4.122
ci_high             3.476  4.164
true_mean           3.289  3.393
naive_bias          0.304  0.923
ipw_bias            0.173  0.750
bias_removed_share  0.432  0.188
ci_covers_truth     False  False

Pre-to-post change  true: 0.104  naive: 0.724  IPW: 0.681


**Your revision**
Version 1 simulated its own population (20,000 users, a 32% response rate and a true mean of 5.526, which is impossible on a 1-5 scale) and named the outcome satisfaction, a column that does not exist in my notebook. Its numbers therefore said nothing about my survey. In Version 2 I changed only the [Iterate] section: I told the model not to simulate data, gave it my DataFrame and column names, named score as the outcome, and gave it the observed and true means and my seed, so its code would run on my data and its estimate could be checked against my known truth. The revision changed the output: version 2 ran on my pre and post DataFrames, used score as the outcome, and returned estimates I could check against my known true means. It also correctly left out s, which would not be observable in a real survey.

**Your check**

The correction did not work. After the redesign, the IPW-corrected mean is 4.144 (95% bootstrap interval 4.122 to 4.164), against a known true mean of 3.393 and an uncorrected survey mean of 4.317. Weighting removed only about 19% of the bias, from +0.923 to +0.750 points, and the interval does not contain the truth. Before the redesign it did a little better, removing about 43% of a smaller bias (IPW 3.462 against a true 3.289), but its interval misses the truth too. The headline result barely moves: the estimated pre-to-post improvement is 0.681 points with IPW, compared with 0.724 uncorrected and a true improvement of 0.104.

This is the expected result. My response mechanism depends on satisfaction itself, so response is missing not at random. The logistic model only sees engagement and support tickets, so it can only correct the part of the selection that runs through them. Within every group of users with the same covariates, the ones who answered are still more satisfied than the ones who did not, and no weights built from those covariates can fix that. Correcting less after the redesign also makes sense: there, response depends far more strongly on satisfaction (b = 2.00 instead of 0.45), so a larger share of the selection is invisible to the covariates.

**In your own words**

response_propensity() standardises engagement_minutes and had_support_ticket, then fits an unpenalised logistic regression on all 50,000 users, predicting the 0/1 responded column, so the model learns from respondents and non-respondents alike; predict_proba then gives every user an estimated probability of answering. ipw_mean() clips those probabilities at 0.001 so no single user can get an extreme weight, gives each respondent a weight of one over their own probability, so a respondent who was unlikely to answer stands in for many similar users who stayed silent, and takes the weighted average of their scores divided by the sum of the weights (the Hajek estimator). The weakness is that the probability depends only on the two covariates, never on satisfaction, so the weights cannot see the main reason people answered.

**One sentence**

Yes, I would use AI for this step again, because it wrote the weighting and bootstrap code quickly and explained the MAR assumption clearly, but only together with my own simulation, since version 1's numbers looked convincing yet came from data it had invented.

### AI Transcript

**Model:** Claude Opus 5.5
**Date:** 2026-10-19

**Prompt, version 1 (as sent):**

[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.


**Reply to version 1 (unedited):**
No dataset came through with your message. So the script includes a simulator that matches the mechanism you described, using your variable names. To run it on your own data, pass your DataFrame to `ipw_mean()` and `bootstrap_ipw()`. The one name I had to invent is the response indicator, `responded`. The code assumes `engagement_minutes` and `had_support_ticket` are known for everyone sampled, which is usually true because they come from product logs, while `satisfaction` is known only for respondents.

**What the code does**

1. It fits an unpenalised logistic regression of `responded` on the two covariates, using the full sample.
2. It weights each respondent by 1/p̂.
3. It computes the self-normalised (Hajek) weighted mean.
4. For the interval, it resamples the whole frame, respondents and non-respondents together, and refits the propensity model on each resample. That way the interval also reflects the uncertainty from estimating the weights.

**Result on the simulated population (n = 20,000, response rate 32%):**

| | Estimate | Bias |
|---|---|---|
| True mean | 5.526 | — |
| Naive respondent mean | 6.795 | +1.270 |
| IPW estimate | 6.509 | +0.983 |
| 95% bootstrap CI | [6.472, 6.544] | misses the truth |

**Which assumption makes the correction valid**

The key assumption is conditional ignorability, also called missing at random (MAR): R ⊥ Y | X. Once you condition on `engagement_minutes` and `had_support_ticket`, whether someone responds must carry no further information about their satisfaction. Formally, P(R=1 | X, Y) = P(R=1 | X).

When that holds, E[R·Y / π(X)] = E[ E[R|X,Y]·Y / π(X) ] = E[Y], so the weighting is exactly unbiased. Two supporting conditions are also needed. The first is positivity: π(X) > 0 everywhere, so no covariate cell is entirely missing. The second is that the logistic model for π(X) is correctly specified.

**What happens when response depends on satisfaction itself**

That is missing not at random (MNAR). The probability that actually matters is P(R=1 | X, Y), but the model can only estimate P(R=1 | X), which averages over Y. As a result, the IPW estimator converges to E_X[ E[Y | X, R=1] ] rather than E_X[ E[Y | X] ].

Because response increases with Y, respondents within every covariate cell are more satisfied than non-respondents in the same cell. So E[Y | X, R=1] > E[Y | X], and the upward bias remains.

The weights do correct the part of the selection that runs through X. Covariates that predict satisfaction act as partial proxies for it. Here that removed about 23% of the bias. The share removed grows with the fraction of satisfaction's variance the covariates explain, and it reaches 100% only if satisfaction is a deterministic function of them.

**Evaluation: will this recover the true mean? No.**

Your mechanism makes response a function of satisfaction itself, so MAR is violated by construction. No model of response on `engagement_minutes` and `had_support_ticket` can produce the correct weights, no matter how well it is specified or how large n is. The bias does not shrink as the sample grows.

The bootstrap makes this worse in a specific way. It measures only sampling variability, so it produces a tight interval around the wrong value. In the simulation, the 95% interval excludes the truth entirely.

**What would help instead**

- **Sensitivity analysis:** apply a delta or pattern-mixture adjustment, with an assumed shift in non-respondents' satisfaction, and report how the estimate changes across plausible shifts.
- **Follow-up of non-respondents:** use a randomised incentive or a double-sampling design to measure a subsample of non-respondents.
- **Selection models:** use a Heckman-type model or a shadow variable, which identify the mean only under their own untestable assumptions.


Inverse-probability weighting (IPW) for survey non-response.

Expected columns in the sample-frame DataFrame (ALL sampled users,
respondents and non-respondents alike):
    satisfaction        -- observed only where responded == 1 (NaN otherwise)
    engagement_minutes  -- known for everyone (product logs)
    had_support_ticket  -- known for everyone (0/1)
    responded           -- 1 if the user answered the survey, else 0

Replace `simulate_population()` with your own DataFrame if you have one;
`ipw_mean()` and `bootstrap_ipw()` only need the four columns above.

~~~
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

COVARIATES = ["engagement_minutes", "had_support_ticket"]


# --------------------------------------------------------------------------
# Simulation matching the described mechanism: P(respond) rises with
# satisfaction ITSELF (MNAR). Covariates affect satisfaction, not response
# directly.
# --------------------------------------------------------------------------
def simulate_population(n=20_000, seed=0):
    rng = np.random.default_rng(seed)
    engagement_minutes = rng.lognormal(mean=3.0, sigma=0.6, size=n)
    p_ticket = 1 / (1 + np.exp(-(-0.5 - 0.03 * (engagement_minutes - 20))))
    had_support_ticket = rng.binomial(1, p_ticket)
    satisfaction = (
        5.0
        + 0.04 * engagement_minutes
        - 1.2 * had_support_ticket
        + rng.normal(0, 1.5, size=n)
    )
    p_respond = 1 / (1 + np.exp(-(-1.0 + 0.8 * (satisfaction - satisfaction.mean()))))
    responded = rng.binomial(1, p_respond)

    df = pd.DataFrame({
        "satisfaction_true": satisfaction,  # oracle column, unknowable in practice
        "engagement_minutes": engagement_minutes,
        "had_support_ticket": had_support_ticket,
        "responded": responded,
    })
    df["satisfaction"] = np.where(responded == 1, satisfaction, np.nan)
    return df


# --------------------------------------------------------------------------
# IPW estimator
# --------------------------------------------------------------------------
def fit_response_propensity(df):
    """Logistic model of P(responded = 1 | covariates), fit on the full frame."""
    X = df[COVARIATES].to_numpy(dtype=float)
    X = (X - X.mean(axis=0)) / X.std(axis=0)  # scaling helps the solver
    # Effectively unpenalised; C=1e10 works across old and new sklearn versions
    model = LogisticRegression(C=1e10, max_iter=1000)
    model.fit(X, df["responded"].to_numpy())
    return model.predict_proba(X)[:, 1]


def ipw_mean(df, clip=1e-3):
    """Hajek (self-normalised) IPW mean of satisfaction among respondents."""
    p_hat = np.clip(fit_response_propensity(df), clip, 1.0)
    resp = df["responded"].to_numpy() == 1
    w = 1.0 / p_hat[resp]
    y = df.loc[resp, "satisfaction"].to_numpy()
    return np.sum(w * y) / np.sum(w)


def bootstrap_ipw(df, n_boot=1000, alpha=0.05, seed=1):
    """
    Nonparametric bootstrap: resample the WHOLE frame (respondents and
    non-respondents) and refit the propensity model each time, so the
    interval reflects uncertainty in the estimated weights too.
    """
    rng = np.random.default_rng(seed)
    n = len(df)
    estimates = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        estimates[b] = ipw_mean(df.iloc[idx].reset_index(drop=True))
    lo, hi = np.quantile(estimates, [alpha / 2, 1 - alpha / 2])
    return lo, hi, estimates


# --------------------------------------------------------------------------
if __name__ == "__main__":
    df = simulate_population()

    true_mean = df["satisfaction_true"].mean()
    naive_mean = df["satisfaction"].mean()  # pandas skips NaN
    ipw_est = ipw_mean(df)
    lo, hi, _ = bootstrap_ipw(df, n_boot=500)

    print(f"Response rate        : {df['responded'].mean():.3f}")
    print(f"True population mean : {true_mean:.3f}")
    print(f"Naive respondent mean: {naive_mean:.3f}  (bias {naive_mean - true_mean:+.3f})")
    print(f"IPW estimate         : {ipw_est:.3f}  (bias {ipw_est - true_mean:+.3f})")
    print(f"95% bootstrap CI     : [{lo:.3f}, {hi:.3f}]")
    print(f"CI covers truth?     : {lo <= true_mean <= hi}")
    print(f"Share of naive bias removed: "
          f"{1 - (ipw_est - true_mean) / (naive_mean - true_mean):.1%}")
~~~

**What I changed:** I rewrote only the [Iterate] section, to give the model my DataFrames, column names, observed and true means, and seed, and to tell it not to simulate new data.

**Prompt, version 2 (as sent):**
```
[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Do not simulate new data; use my existing DataFrames. I already have two pandas DataFrames in my notebook, pre and post, each with 50,000 rows and these columns: engagement_minutes, engagement_z, had_support_ticket, s (latent satisfaction), score (the 1-5 survey score) and responded (True if the user answered). The outcome is score, not satisfaction. Correct the observed mean of score among respondents in post, which is 4.317; the true population mean of score in post is 3.393, and in pre the observed mean is 3.593 and the true mean is 3.289. Use np.random.default_rng(3916) for the bootstrap. Return the weighted estimate for pre and post and a bootstrap interval that refits the response model inside each resample.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.
```

**Reply to version 2 (unedited):**
```
The script in ipw_pre_post.py runs directly on your existing `pre` and `post` DataFrames and does not simulate anything. I don't have access to your notebook, so I can't report your actual weighted estimates or intervals. Running the script will print them next to the figures you quoted (pre: observed 3.593, true 3.289; post: observed 4.317, true 3.393), along with the bias, the share of bias removed, and whether each interval covers the truth. I checked that it runs end to end on throwaway mock frames with your column layout.

**Design choices**

- **Response model:** a logistic regression of `responded` on `engagement_minutes` and `had_support_ticket`, unpenalised and fit on all 50,000 rows of each wave. I left out `engagement_z` because it is just a rescaling of `engagement_minutes`, and including both would make the model collinear.
- **Variable `s` is not used:** in a real survey you could not observe it, so using it would cheat the correction.
- **Estimator:** the Hajek (self-normalised) weighted mean of `score`, using respondents' scores only.
- **Bootstrap:** a single `np.random.default_rng(3916)` is used for pre and then post, with 1,000 resamples each. Each resample draws from the full frame, respondents and non-respondents together, and refits the response model before reweighting. The interval is the 2.5th to 97.5th percentile of the resampled estimates.

**Which assumption makes the correction valid**

The key assumption is conditional ignorability, also called missing at random (MAR), for the outcome you are estimating:

P(responded = 1 | X, score) = P(responded = 1 | X)

That means that once you condition on engagement and support tickets, responding must tell you nothing more about `score`. If that holds, E[R·Y/π(X)] = E[Y], and the weighted mean is consistent. Two further conditions are needed. Positivity: every covariate pattern has π(X) > 0. Correct specification: the logistic model gets π(X) right.

**What happens when response depends on satisfaction itself**

In your data, response is driven by `s`, and `score` is a coarsened version of `s`. So given X, respondents still have higher `s`, and therefore higher `score`, than non-respondents with the same covariates. That is, E[score | X, R=1] > E[score | X].

Because of this, IPW converges to E_X[ E[score | X, R=1] ], not to the population mean. The weights rebalance respondents across covariate cells. They cannot fix the selection that happens inside each cell, because the model never sees the variable doing the selecting.

The covariates help only to the extent that they predict `s`. Whatever part of `s` they don't explain remains as uncorrected selection, and the bias stays upward.

Two consequences follow:

- **More data won't fix it.** The bias does not shrink with n.
- **The interval will be misleading.** The bootstrap captures only sampling noise, so it will be narrow and centred on the biased value.

**Evaluation: will this correction recover the true mean? No.**

Your mechanism is missing not at random by construction: response depends on `s`, which is not among the covariates. No logistic model on `engagement_minutes` and `had_support_ticket` produces the right weights. You should expect:

- **Both estimates stay high:** the pre and post estimates will sit only partly below the observed means of 3.593 and 4.317, still above the truths of 3.289 and 3.393.
- **The intervals miss the truth:** they will very likely exclude the true means.
- **Larger residual bias in post:** the naive bias is about three times larger there (0.924 versus 0.304), so post will likely show more remaining bias.

That last point matters for any pre/post comparison. The true change is only +0.104, but the selection bias differs between waves. The reweighted change will therefore still be inflated, and most of it will reflect a change in who responds rather than a real change in satisfaction.

If you need defensible numbers, report a sensitivity analysis over plausible gaps between respondents and non-respondents in each wave, or collect `score` from a random follow-up subsample of non-respondents.
````

~~~
"""
IPW non-response correction for `score`, run on the existing `pre` and `post`
DataFrames (no simulation). Paste into the notebook after `pre` and `post` exist.

Columns used: engagement_minutes, had_support_ticket, score, responded.
`engagement_z` is a rescaling of engagement_minutes, so it is left out
(including both would make the design matrix collinear).
`s` is never used: it is the latent quantity driving response and would not
be observable in a real survey.
"""
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

COVARIATES = ["engagement_minutes", "had_support_ticket"]
N_BOOT = 1000
ALPHA = 0.05


def response_propensity(df):
    """Fit P(responded | covariates) on the full frame; return fitted probabilities."""
    X = df[COVARIATES].astype(float).to_numpy()
    X = (X - X.mean(axis=0)) / X.std(axis=0)
    r = df["responded"].astype(int).to_numpy()
    model = LogisticRegression(C=1e10, max_iter=1000)  # effectively unpenalised
    model.fit(X, r)
    return model.predict_proba(X)[:, 1]


def ipw_mean(df, clip=1e-3):
    """Hajek IPW mean of score, using respondents' scores only."""
    p = np.clip(response_propensity(df), clip, 1.0)
    resp = df["responded"].astype(bool).to_numpy()
    w = 1.0 / p[resp]
    y = df["score"].to_numpy(dtype=float)[resp]
    return np.sum(w * y) / np.sum(w)


def bootstrap_ipw(df, rng, n_boot=N_BOOT, alpha=ALPHA):
    """Resample the whole frame and refit the response model in every resample."""
    n = len(df)
    est = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        est[b] = ipw_mean(df.iloc[idx])
    lo, hi = np.quantile(est, [alpha / 2, 1 - alpha / 2])
    return lo, hi, est


# One generator, used for pre first and then post, so the run is reproducible.
rng = np.random.default_rng(3916)

rows = []
for name, df in [("pre", pre), ("post", post)]:
    resp = df["responded"].astype(bool)
    observed = df.loc[resp, "score"].mean()
    true = df["score"].mean()  # full-population mean, known only in simulation
    est = ipw_mean(df)
    lo, hi, _ = bootstrap_ipw(df, rng)
    rows.append({
        "wave": name,
        "response_rate": resp.mean(),
        "observed_mean": observed,
        "ipw_estimate": est,
        "ci_low": lo,
        "ci_high": hi,
        "true_mean": true,
        "naive_bias": observed - true,
        "ipw_bias": est - true,
        "bias_removed_share": 1 - (est - true) / (observed - true),
        "ci_covers_truth": lo <= true <= hi,
    })

results = pd.DataFrame(rows).set_index("wave")
pd.set_option("display.float_format", "{:.3f}".format)
print(results.T)

# Expected sanity checks against the figures you quoted:
#   pre : observed 3.593, true 3.289
#   post: observed 4.317, true 3.393
num = ["true_mean", "observed_mean", "ipw_estimate"]
change = results.loc["post", num] - results.loc["pre", num]
print(f"\nPre-to-post change  true: {change['true_mean']:.3f}  "
      f"naive: {change['observed_mean']:.3f}  IPW: {change['ipw_estimate']:.3f}")
~~~

### Task 4.2: The slide

**Onboarding redesign: what the satisfaction survey can and cannot tell us**

The survey does **not** show that satisfaction rose from 3.6 to 4.3. After the redesign, only 15% of users answered (down from 27%), and the new prompt is answered far more often by satisfied users. Most of the 0.7-point jump reflects who chose to respond, not how users feel.

The survey **does** tell us that respondents became more positive, and the data are consistent with a small real improvement.

**Best estimate of the change: no reliable number yet.** Even after reweighting respondents on engagement and support tickets, the post-redesign estimate is 4.14 (95% CI 4.12 to 4.16), still far above what the full user base would likely report. Adjusting for who answers cannot fix a survey whose response depends on satisfaction itself.

**The fix:** send the survey to a random sample of users, and follow up with non-respondents (for example, with a small incentive), so the people who answer represent everyone.